# 10-1절 연습 문제 풀이

이 노트북은 10-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch10/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 10장 공통 - 트랜스포머 도우미 (본문 10-01~10-03 노트북 참고)
import random as _r, math
from torch.utils.data import Dataset, DataLoader
PAD, SOS, EOS, UNK = '<pad>', '<sos>', '<eos>', '<unk>'

def make_sort_pairs(n=5000, count=10, seed=SEED):
    rng = _r.Random(seed)
    return [(lambda v: (', '.join(map(str, v)), ', '.join(map(str, sorted(v)))))(
             [rng.randint(1, 1000) for _ in range(count)]) for _ in range(n)]

def build_vocab(texts):
    return {t: i for i, t in enumerate([PAD, SOS, EOS, UNK] +
                                       sorted({c for t in texts for c in t}))}

class SeqDataset(Dataset):
    def __init__(self, pairs, sv, tv, sl=60, tl=62):
        self.p, self.sv, self.tv, self.sl, self.tl = pairs, sv, tv, sl, tl
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        s, t = self.p[i]
        src = [self.sv.get(c, self.sv[UNK]) for c in s][:self.sl]
        src += [self.sv[PAD]] * (self.sl - len(src))
        tgt = [self.tv[SOS]] + [self.tv.get(c, self.tv[UNK]) for c in t] + [self.tv[EOS]]
        tgt = tgt[:self.tl] + [self.tv[PAD]] * (self.tl - len(tgt))
        return torch.tensor(src), torch.tensor(tgt)

class DateConverterTransformer(nn.Module):
    def __init__(self, sv, tv, d_model=128, nhead=4, layers=2, max_len=64):
        super().__init__()
        self.src_embedding = nn.Embedding(sv, d_model, padding_idx=0)
        self.tgt_embedding = nn.Embedding(tv, d_model, padding_idx=0)
        self.pos_encoding = nn.Embedding(max_len, d_model)
        self.transformer = nn.Transformer(d_model, nhead, layers, layers,
                                          d_model * 4, batch_first=True)
        self.fc = nn.Linear(d_model, tv)
        self.dropout = nn.Dropout(0.1)
    def encode_pos(self, x, emb):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.dropout(emb(x) + self.pos_encoding(pos))
    def forward(self, src, tgt):
        src_mask = (src == 0)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            tgt.size(1), device=tgt.device)
        out = self.transformer(self.encode_pos(src, self.src_embedding),
                               self.encode_pos(tgt, self.tgt_embedding),
                               tgt_mask=tgt_mask,
                               src_key_padding_mask=src_mask,
                               memory_key_padding_mask=src_mask)
        return self.fc(out)

def train_tf(model, loader, epochs=20, lr=1e-3, scheduler_fn=None):
    model = model.to(device)
    crit = nn.CrossEntropyLoss(ignore_index=0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        if scheduler_fn:
            for g in opt.param_groups: g['lr'] = scheduler_fn(e)
        model.train(); tot = n = 0
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            out = model(src, tgt[:, :-1])
            loss = crit(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); n += 1
        if e % 5 == 0 or e == 1: print(f'  {e}/{epochs} 손실 {tot / n:.4f}')
    return model

## 연습 10-1

1에서 1,000 사이의 정수 10개를 무작위로 뽑아 쉼표로 구분한 문자열(예: 5, 724, 223, 695, ...)을 입력하면, 오름차순으로 정렬한 문자열(예: 5, 223, 695, 724, ...)을 생성하는 트랜스포머 모델을 만들어 보자. 이 문제는 9장 [연습 문제 9-9], [연습 문제 9-11]을 트랜스포머 모델로 만들어 보는 문제다.

In [ ]:
pairs = make_sort_pairs(5000)
sv = tv = build_vocab([s for s, _ in pairs] + [t for _, t in pairs])
rev = {i: t for t, i in tv.items()}
loader = DataLoader(SeqDataset(pairs, sv, tv), batch_size=64, shuffle=True)
torch.manual_seed(SEED)
model = train_tf(DateConverterTransformer(len(sv), len(tv)), loader, epochs=20)

트랜스포머는 어텐션으로 입력 전체를 매 시점 참조하므로, 콘텍스트 벡터 병목이 있던 9장 Seq2Seq보다 정렬 과제를 훨씬 잘 푼다. 게다가 순환 구조가 없어 **한 배치를 병렬로 처리**하므로 학습도 빠르다.

## 연습 10-2

[표 10-4]를 보면 멀티헤드 어텐션의 헤드 수(num_heads)에 따라 모델 3과 모델 4의 성능이 크게 차이 나는 반면 파라미터 수와 에포크당 학습 시간은 거의 차이가 없다.

헤드 수를 늘려도 파라미터 수가 바뀌지 않는 이유를 본문에서 찾아 설명해 보자.

헤드 수에 따라 모델 성능과 에포크당 학습 시간이 어떻게 바뀌는지 가능한 모든 헤드 수로 모델을 학습해 확인해 보자.

In [ ]:
for nhead in (1, 2, 4, 8):
    torch.manual_seed(SEED)
    m = DateConverterTransformer(len(sv), len(tv), d_model=128, nhead=nhead)
    print(f'헤드 {nhead}개: 파라미터 {sum(p.numel() for p in m.parameters()):,}개')

**헤드 수를 늘려도 파라미터가 늘지 않는 이유**: 멀티헤드 어텐션은 `d_model` 차원을 헤드 수만큼 **쪼개어** 나눠 쓴다. 헤드가 4개면 각 헤드가 32차원씩 담당한다. Q·K·V 투영 행렬의 전체 크기는 `d_model × d_model`로 고정이므로 헤드 수와 무관하다.

성능은 헤드가 여럿일 때 좋아진다. 서로 다른 부분 공간에서 **여러 종류의 관계를 동시에** 볼 수 있기 때문이다. 다만 `d_model`을 나누므로 헤드가 지나치게 많으면 헤드당 차원이 너무 작아져 오히려 나빠진다.

## 연습 10-3

트랜스포머 모델은 학습 초기에 파라미터 초깃값과 학습률에 민감해 학습이 불안정해지기 쉽다. 이를 완화하려고 처음에는 낮은 학습률로 시작해 점차 높이는 학습률 워밍업 방식을 사용하기도 한다.

헤드 수가 1인 트랜스포머 모델에 다음과 같이 간소화한 학습률 워밍업 방법을 적용하여 노이즈가 추가된 데이터를 학습하고 결과를 비교해 보자.

첫 10 에포크의 학습률: 1e-4

이후 에포크의 학습률: 1e-3

참고로 다음 방법으로 학습 도중에 Adam 옵티마이저의 학습률을 바꿀 수 있다.

*코드 10-7 학습 중간에 Adam 옵티마이저의 학습률 변경*

```python
for param_group in optimizer.param_groups:
    param_group['lr'] = NEW_LR
```

In [ ]:
def warmup(epoch, base=1e-3, warm=10):
    return base * epoch / warm if epoch <= warm else base

torch.manual_seed(SEED)
print('[워밍업 없음]')
train_tf(DateConverterTransformer(len(sv), len(tv), nhead=1), loader, epochs=20)
torch.manual_seed(SEED)
print('[워밍업 적용]')
train_tf(DateConverterTransformer(len(sv), len(tv), nhead=1), loader, epochs=20,
         scheduler_fn=warmup)

트랜스포머는 학습 초기에 어텐션 가중치가 무작위라 큰 학습률을 주면 파라미터가 크게 흔들려 발산하기 쉽다. 워밍업은 **처음에 작은 보폭으로 시작해** 어텐션이 어느 정도 자리를 잡은 뒤 학습률을 올린다. 원 논문도 워밍업 후 감소하는 스케줄을 사용했다.

## 연습 10-4

[도전 문제] 학습이 끝난 DateConverterTransformer 모델을 사전 학습 모델로 사용해 영어 형식의 날짜 문자열을 한국어 날짜 문자열로 변환하는 모델을 만들어 보자([연습 문제 9-13] 참고).

### 풀이

[연습 문제 9-13]과 같은 전이 학습을 트랜스포머로 수행한다. 절차는 다음과 같다.

1. 영어 날짜 변환 모델을 충분히 학습해 **사전 학습 모델**로 삼는다.
2. 출력 어휘가 한글로 바뀌므로 **`tgt_embedding`과 `fc`(출력층)는 새로 만든다**. 나머지(인코더, 위치 인코딩, 디코더 본체)는 그대로 가져온다.
3. 작은 학습률로 미세 조정한다.

인코더가 이미 '영어 날짜 문자열 읽는 법'을 알고 있으므로 적은 데이터로도 빠르게 수렴한다.

In [ ]:
import copy
# 사전 학습(영어 날짜 정렬 과제로 대체) 후 출력 쪽만 교체하는 예
pre = model                                  # 위에서 학습한 모델을 사전 학습 모델로 사용
kor_tv = build_vocab(['이천이십육년 이월 일일'])
transfer = copy.deepcopy(pre)
transfer.tgt_embedding = nn.Embedding(len(kor_tv), 128, padding_idx=0).to(device)
transfer.fc = nn.Linear(128, len(kor_tv)).to(device)
print(f'교체한 출력 어휘 {len(kor_tv)}개 / '
      f'유지한 인코더 파라미터 '
      f'{sum(p.numel() for p in transfer.transformer.encoder.parameters()):,}개')

## 연습 10-5

[도전 문제] 학습 가능한 위치 인코딩 대신 원래 트랜스포머 논문에 등장하는 사인/코사인 기반 위치 인코딩을 직접 구현해 PositionalEncoding 클래스를 대체해 보자. 이후 9장과 10장 깃허브 예제 노트북을 참조해 깨끗한 데이터와 노이즈가 추가된 데이터로 데이터셋을 만든 후, 이를 사용해 모델을 학습해 보며 입력에 노이즈가 있을 때와 없을 때 두 위치 인코딩 방식에 따라 결과가 어떻게 달라지는지 확인해 보자.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)          # 학습 대상이 아님
    def forward(self, x):                        # x: (B, T, d_model)
        return x + self.pe[:x.size(1)].unsqueeze(0)

pe = SinusoidalPositionalEncoding(128)
x = torch.zeros(1, 20, 128)
print(f'위치 인코딩 적용: {tuple(pe(x).shape)}, 학습 파라미터 '
      f'{sum(p.numel() for p in pe.parameters())}개')

class SinTransformer(DateConverterTransformer):
    def __init__(self, sv, tv, **kw):
        super().__init__(sv, tv, **kw)
        self.pos = SinusoidalPositionalEncoding(kw.get('d_model', 128))
    def encode_pos(self, x, emb):
        return self.dropout(self.pos(emb(x)))

torch.manual_seed(SEED)
train_tf(SinTransformer(len(sv), len(tv)), loader, epochs=20)

사인·코사인 위치 인코딩은 **학습 파라미터가 없다**(`register_buffer`로 등록). 주파수가 다른 사인·코사인을 겹쳐 위치마다 고유한 패턴을 만들고, 학습에서 본 적 없는 긴 입력에도 값을 계산할 수 있다는 장점이 있다.

학습형 위치 임베딩은 데이터에 맞는 표현을 배우지만 `max_len`을 넘는 위치는 다룰 수 없다. 짧고 길이가 일정한 이 과제에서는 둘의 성능 차이가 크지 않다.